# NB71 — Charge Sector Decomposition

NB70 established the dynamical VEV bridge for the **physical sector** ($a_5 = 0$):

$$m_s/m_d = R_{4,\text{quark}}^{\,\varphi(210)/(2\pi)} = 19.92 \quad (-0.032\sigma)$$

But Z*₂₁₀ has **four** charge sectors indexed by $a_5 \in \{0,1,2,3\}$, each with a different SPEC5 eigenvalue and different SM interpretation (NB62):

| $a_5$ idx | Raw $a_5$ | SPEC5 | SM Interpretation | Interference |
|-----------|-----------|-------|-------------------|-------------|
| 0 | 1 | 0 | Down quarks + charged leptons | Constructive (3+1) |
| 1 | 2 | 2 | Isospin doublet member | Mixed |
| 2 | 4 | 4 | Tower-protected | Zero inter-gen split |
| 3 | 3 | 2 | Isospin conjugate | Mixed |

**Key questions**:
1. Do R₄ conjugate pair ratios **differ** across $a_5$ sectors?
2. Does the $a_5 = 1,3$ sector (SPEC5=2 extra) explain the up-type quark mass hierarchy ($m_c/m_u \approx 580$)?
3. Is the $a_5 = 2$ sector (tower-protected) dynamically flat — consistent with near-degenerate neutrino masses?

**SM targets**: $m_s/m_d \approx 20$, $m_c/m_u \approx 580$, $(m_c/m_u)/(m_s/m_d) \approx 29$.

In [1]:
# -- NB71 Setup (reuses NB70 ODE infrastructure) --
import sys, numpy as np
from pathlib import Path
from scipy.integrate import solve_ivp
from collections import defaultdict
from math import gcd
import itertools

ROOT = Path.cwd().parent
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))
from solenoid_algebra import SA

PRIMES = [2, 3, 5, 7]
PRIMORIALS = [1, 2, 6, 30, 210]
N = 210
PHI = 48
OMEGA = 2 * np.pi
RHO = 1 / np.sqrt(210)
EPS = RHO
N_LEVELS = 5
ALL_BRANCHES = list(itertools.product(*[range(p) for p in PRIMES]))

# Tower eigenvalue spectra (NB56/62)
SPEC3 = [0, 2]
SPEC5 = [0, 2, 4, 2]
SPEC7 = [0, 1, 3, 4, 3, 1]

# Discrete log tables
DLOG = {3: {1:0, 2:1}, 5: {1:0, 2:1, 4:2, 3:3}, 7: {1:0, 3:1, 2:2, 6:3, 4:4, 5:5}}
# Inverse tables (dlog index -> raw residue)
RAW_A3 = {0: 1, 1: 2}
RAW_A5 = {0: 1, 1: 2, 2: 4, 3: 3}
RAW_A7 = {0: 1, 1: 3, 2: 2, 3: 6, 4: 4, 5: 5}

# SM targets
M_MU_ME  = 206.768
M_S_MD   = 20.0;  M_S_MD_ERR = 2.5
M_C_MU   = 11.8;  M_C_MU_ERR = 2.0    # m_c/m_u ~ 11.8 (using constituent), or ~580 (current quark)
# PDG current quark masses: m_u = 2.16 MeV, m_c = 1270 MeV -> m_c/m_u = 588
M_C_MU_CURRENT = 588.0; M_C_MU_CURRENT_ERR = 100.0
M_TAU_MU  = 16.817
M_B_MS    = 49.0;  M_B_MS_ERR = 8.0    # m_b/m_s ~ 49

# NB70 amplification exponent
X_AMP = PHI / (2 * np.pi)  # 48/(2pi) ~ 7.6394
LOG_RATIO_ALG = 3 * (RHO + 1) / (RHO + np.sqrt(3))

# ODE infrastructure
def make_ode_lr(eps, kappa):
    def ode(t, theta):
        d = np.zeros(N_LEVELS)
        d[0] = OMEGA
        for k in range(1, N_LEVELS):
            p = PRIMES[k-1]
            R_k = p * theta[k] - theta[k-1]
            d[k] = OMEGA / PRIMORIALS[k] + eps * np.sin(theta[k-1]) / p - kappa * R_k / p
        return d
    return ode

def branch_ic(branch):
    theta = np.zeros(N_LEVELS)
    for k in range(4):
        theta[k+1] = (theta[k] + 2*np.pi*branch[k]) / PRIMES[k]
    return theta

def integrate_and_section(ode_fn, theta0, T=5000):
    n_pts = max(500_000, int(T * 200))
    sol = solve_ivp(ode_fn, [0, T], theta0,
                    t_eval=np.linspace(0, T, n_pts),
                    method='RK45', rtol=1e-10, atol=1e-12)
    th0_mod = np.mod(sol.y[0], 2*np.pi)
    crossings = np.where(np.diff(th0_mod) < -np.pi)[0]
    sections = sol.y[:, crossings]
    n_cross = sections.shape[1]
    R = np.zeros((4, n_cross))
    for k in range(4):
        p = PRIMES[k]
        raw = p * sections[k+1] - sections[k]
        R[k] = np.mod(raw, 2*np.pi)
        R[k][R[k] > np.pi] -= 2*np.pi
    return sections, R, n_cross

print("Setup complete")
print(f"  rho = eps = kappa = 1/sqrt(210) = {RHO:.6f}")
print(f"  Amplification exponent x = phi(210)/(2pi) = {X_AMP:.4f}")
print(f"  SM targets: m_s/m_d = {M_S_MD}, m_c/m_u(current) = {M_C_MU_CURRENT}")
print(f"  SPEC5 = {SPEC5}  (a5_idx -> tower eigenvalue from p=5)")

Setup complete
  rho = eps = kappa = 1/sqrt(210) = 0.069007
  Amplification exponent x = phi(210)/(2pi) = 7.6394
  SM targets: m_s/m_d = 20.0, m_c/m_u(current) = 588.0
  SPEC5 = [0, 2, 4, 2]  (a5_idx -> tower eigenvalue from p=5)


## Data Collection — All Four Charge Sectors

Collect RMS covering residuals at **all 4 R-levels** for **all 4 $a_5$ sectors**, resolved by chirality ($a_3$) and generation index ($a_7$). This extends NB70's physical-sector-only collection to the full Z₄ charge decomposition.

Accumulator: `sector_rms[a5_idx][a3_idx][a7_star][level]` — one entry per charge sector, chirality, generation star index, and covering level.

In [2]:
# -- Data Collection: all 4 a5 sectors, all levels, chirality-resolved --
N_BR = 50
np.random.seed(42)
sample = [ALL_BRANCHES[i] for i in np.random.choice(len(ALL_BRANCHES), N_BR, replace=False)]

# sector_accum[a5_idx][a3_idx][a7_star][level] = [sum_sq, count]
sector_accum = {}
for a5i in range(4):
    sector_accum[a5i] = {}
    for a3i in range(2):
        sector_accum[a5i][a3i] = {}
        for a7s in range(6):
            sector_accum[a5i][a3i][a7s] = {lev: [0.0, 0] for lev in range(4)}

for ib, br in enumerate(sample):
    theta0 = branch_ic(br)
    ode = make_ode_lr(EPS, EPS)
    _, R, n_cross = integrate_and_section(ode, theta0)
    
    for i in range(n_cross):
        if gcd(i, N) != 1:
            continue
        a3_raw, a5_raw, a7_raw = i % 3, i % 5, i % 7
        if a3_raw == 0 or a5_raw == 0 or a7_raw == 0:
            continue
        
        a3_idx = DLOG[3][a3_raw]
        a5_idx = DLOG[5][a5_raw]
        a7_star = DLOG[7][a7_raw]
        
        for level in range(4):
            bucket = sector_accum[a5_idx][a3_idx][a7_star][level]
            bucket[0] += R[level][i] ** 2
            bucket[1] += 1
    
    if (ib + 1) % 10 == 0:
        print(f"  Branch {ib+1}/{N_BR}")

# Compute RMS
sector_rms = {}
for a5i in range(4):
    sector_rms[a5i] = {}
    for a3i in range(2):
        sector_rms[a5i][a3i] = {}
        for a7s in range(6):
            sector_rms[a5i][a3i][a7s] = {}
            for lev in range(4):
                sq, cnt = sector_accum[a5i][a3i][a7s][lev]
                sector_rms[a5i][a3i][a7s][lev] = np.sqrt(sq / cnt) if cnt > 0 else 0.0

# Quick summary
print(f"\nData collected for {N_BR} branches.")
a5_labels = ['down+lepton', 'isospin-1', 'tower-prot', 'isospin-3']
for a5i in range(4):
    total_entries = sum(sector_accum[a5i][a3i][a7s][3][1]
                        for a3i in range(2) for a7s in range(6))
    print(f"  a5={a5i} ({a5_labels[a5i]:>12}): {total_entries:>6} R4 entries")

  Branch 10/50
  Branch 20/50
  Branch 30/50
  Branch 40/50
  Branch 50/50

Data collected for 50 branches.
  a5=0 ( down+lepton):  14300 R4 entries
  a5=1 (   isospin-1):  14300 R4 entries
  a5=2 (  tower-prot):  14226 R4 entries
  a5=3 (   isospin-3):  14300 R4 entries


## Test 1: R₄ Sector Profiles

Display the R₄ RMS values for all 4 charge sectors, both chiralities, and all 6 $a_7$ star indices. Look for:
- Does $a_5 = 0$ (physical) match NB70 values?
- Do $a_5 = 1,3$ (isospin doublet) show different conjugate pair ratios?
- Is $a_5 = 2$ (tower-protected) dynamically flat?

In [3]:
# -- Test 1: R4 Sector Profiles --
print("TEST 1: R4 SECTOR PROFILES")
print("=" * 80)

a5_labels = ['a5=0 down+lep', 'a5=1 isospin', 'a5=2 tower-P', 'a5=3 isospin*']
chir_labels = ['L (a3=0)', 'R (a3=1)']

for a5i in range(4):
    print(f"\n--- {a5_labels[a5i]} (SPEC5={SPEC5[a5i]}) ---")
    for a3i in range(2):
        vals = [sector_rms[a5i][a3i][a7s][3] for a7s in range(6)]
        print(f"  {chir_labels[a3i]}: " +
              "  ".join(f"a7*={a7s}:{vals[a7s]:.4f}" for a7s in range(6)))
    
    # Generation grouping: gen0={0,3}, gen1={1,4}, gen2={2,5}
    print(f"  Generation means (L+R avg at R4):")
    for gen, (s1, s2) in enumerate([(0,3), (1,4), (2,5)]):
        mean_val = np.mean([sector_rms[a5i][a3i][s][3]
                           for a3i in range(2) for s in (s1, s2)])
        print(f"    Gen{gen} (a7*={s1},{s2}): {mean_val:.4f}")

# Cross-sector comparison at R4 level (global means)
print(f"\n\n--- CROSS-SECTOR SUMMARY (R4 global mean) ---")
print(f"{'Sector':<20} {'Mean R4':>10} {'Std R4':>10} {'Max/Min':>10}")
for a5i in range(4):
    all_vals = [sector_rms[a5i][a3i][a7s][3]
                for a3i in range(2) for a7s in range(6)]
    mn, sd = np.mean(all_vals), np.std(all_vals)
    ratio = max(all_vals)/min(all_vals) if min(all_vals) > 0 else 0
    print(f"{a5_labels[a5i]:<20} {mn:10.4f} {sd:10.4f} {ratio:10.4f}")

TEST 1: R4 SECTOR PROFILES

--- a5=0 down+lep (SPEC5=0) ---
  L (a3=0): a7*=0:0.4778  a7*=1:0.4877  a7*=2:0.2398  a7*=3:0.2404  a7*=4:0.2403  a7*=5:0.2464
  R (a3=1): a7*=0:0.3300  a7*=1:0.3136  a7*=2:0.3112  a7*=3:0.5273  a7*=4:0.4604  a7*=5:0.3116
  Generation means (L+R avg at R4):
    Gen0 (a7*=0,3): 0.3939
    Gen1 (a7*=1,4): 0.3755
    Gen2 (a7*=2,5): 0.2772

--- a5=1 isospin (SPEC5=2) ---
  L (a3=0): a7*=0:0.1208  a7*=1:0.1204  a7*=2:0.4783  a7*=3:0.1240  a7*=4:0.1597  a7*=5:0.1203
  R (a3=1): a7*=0:0.1960  a7*=1:0.4177  a7*=2:0.1976  a7*=3:0.1961  a7*=4:0.1963  a7*=5:0.3438
  Generation means (L+R avg at R4):
    Gen0 (a7*=0,3): 0.1592
    Gen1 (a7*=1,4): 0.2235
    Gen2 (a7*=2,5): 0.2850

--- a5=2 tower-P (SPEC5=4) ---
  L (a3=0): a7*=0:0.0565  a7*=1:0.0564  a7*=2:0.0735  a7*=3:0.0567  a7*=4:0.0581  a7*=5:0.3685
  R (a3=1): a7*=0:0.4887  a7*=1:0.2888  a7*=2:0.2842  a7*=3:0.2844  a7*=4:0.2843  a7*=5:0.2809
  Generation means (L+R avg at R4):
    Gen0 (a7*=0,3): 0.2216
    Gen1 

## Test 2: Conjugate Pair Ratios Across Sectors

NB70 used the CP-active conjugate pairs in the physical sector ($a_5=0$):
- **Lepton** (L, CP=1): $a_7^* = 1$ vs $a_7^* = 5$ (SPEC7=1)
- **Quark** (R, CP=0): $a_7^* = 4$ vs $a_7^* = 2$ (SPEC7=3)

Now compute the **same** conjugate pair ratios for all four $a_5$ sectors. The CP-active pairs are the inter-generation pairs where NB69 showed CP-selective breaking (L→CP=1 only, R→CP=0 only).

**Hypothesis**: $a_5 = 1,3$ sectors have enhanced R₃ or R₄ ratios relative to $a_5 = 0$, reflecting their extra SPEC5 contribution to E₂.

In [4]:
# -- Test 2: Conjugate Pair Ratios Per Sector --
print("TEST 2: CONJUGATE PAIR RATIOS (CP-ACTIVE PAIRS)")
print("=" * 80)

# CP-active pairs (from NB69):
#   Lepton: L-chirality (a3_idx=0), a7*=1 vs a7*=5 (SPEC7=1)
#   Quark:  R-chirality (a3_idx=1), a7*=4 vs a7*=2 (SPEC7=3)

cp_pairs = {
    'LEPTON': (0, 1, 5),  # (a3_idx, a7_star_gen1, a7_star_gen2)
    'QUARK':  (1, 4, 2),
}

print(f"\n{'':>14} {'R1':>8} {'R2':>8} {'R3':>8} {'R4':>8}  {'R4^x':>10}  SPEC5")
print("-" * 80)

# Store ratios for later analysis
sector_ratios = {}  # sector_ratios[a5i][particle] = [R1_ratio, ..., R4_ratio]
sector_pred = {}    # sector_pred[a5i][particle] = R4^x prediction

for a5i in range(4):
    sector_ratios[a5i] = {}
    sector_pred[a5i] = {}
    for pname, (a3i, a7_g1, a7_g2) in cp_pairs.items():
        ratios = []
        for lev in range(4):
            v1 = sector_rms[a5i][a3i][a7_g1][lev]
            v2 = sector_rms[a5i][a3i][a7_g2][lev]
            r = v1 / v2 if v2 > 0 else 0
            ratios.append(r)
        sector_ratios[a5i][pname] = ratios
        
        # Bridge prediction: R4^{phi(210)/(2pi)}
        pred = ratios[3] ** X_AMP if ratios[3] > 0 else 0
        sector_pred[a5i][pname] = pred
        
        tag = f"a5={a5i} {pname}"
        r_str = "  ".join(f"{r:7.4f}" for r in ratios)
        print(f"  {tag:<14} {r_str}  {pred:10.2f}  E5={SPEC5[a5i]}")
    print()

# a5=0 cross-check with NB70
print("Cross-check a5=0 vs NB70:")
print(f"  NB70 lepton R4 ratio: 1.9795  NB71: {sector_ratios[0]['LEPTON'][3]:.4f}")
print(f"  NB70 quark  R4 ratio: 1.4794  NB71: {sector_ratios[0]['QUARK'][3]:.4f}")

TEST 2: CONJUGATE PAIR RATIOS (CP-ACTIVE PAIRS)

                     R1       R2       R3       R4        R4^x  SPEC5
--------------------------------------------------------------------------------
  a5=0 LEPTON     6.4537   5.9219   4.2952   1.9795      184.27  E5=0
  a5=0 QUARK     36.7511  20.1672   6.0881   1.4794       19.92  E5=0

  a5=1 LEPTON     0.9998   0.9990   1.0012   1.0006        1.00  E5=2
  a5=1 QUARK      1.0051   1.0112   1.0114   0.9933        0.95  E5=2

  a5=2 LEPTON     0.0474   0.0363   0.1261   0.1532        0.00  E5=4
  a5=2 QUARK      1.0003   1.0016   1.0013   1.0004        1.00  E5=4

  a5=3 LEPTON     1.0618   1.5623   1.5033   1.0449        1.40  E5=2
  a5=3 QUARK      0.1331   0.1785   0.3841   0.5629        0.01  E5=2

Cross-check a5=0 vs NB70:
  NB70 lepton R4 ratio: 1.9795  NB71: 1.9795
  NB70 quark  R4 ratio: 1.4794  NB71: 1.4794


## Test 3: Isospin Sector Mass Prediction

The $a_5 = 1,3$ sectors (isospin doublet) have SPEC5 = 2, adding +2 to the tower eigenvalue $E_2$ compared to the physical sector ($a_5 = 0$, SPEC5 = 0). The NB62 fermion map associates these with **up-type quarks**.

**SM target**: $m_c/m_u \approx 588$ (PDG current quark masses), compared to $m_s/m_d \approx 20$.

**Tests**:
1. Does the isospin sector's R₄ ratio (amplified by $x = \varphi(210)/(2\pi)$) predict a mass ratio near 588?
2. Does the ratio $(m_c/m_u)/(m_s/m_d)$ relate to SPEC5 or R-level differences?
3. Are the $a_5 = 1$ and $a_5 = 3$ sectors symmetric (as NB62's Im₂ cancellation predicts)?

In [5]:
# -- Test 3: Isospin Sector Mass Prediction --
print("TEST 3: ISOSPIN SECTOR MASS PREDICTION")
print("=" * 80)

# The isospin sector should describe up-type quarks
# Both a5=1 and a5=3 have SPEC5=2
# Use the quark conjugate pair (R-chirality)

r4_iso1 = sector_ratios[1]['QUARK'][3]  # a5=1 quark R4 ratio
r4_iso3 = sector_ratios[3]['QUARK'][3]  # a5=3 quark R4 ratio
r4_phys = sector_ratios[0]['QUARK'][3]  # a5=0 quark R4 ratio (NB70)

print(f"\nQuark R4 conjugate pair ratios:")
print(f"  Physical (a5=0, SPEC5=0): {r4_phys:.4f}  -> R4^x = {r4_phys**X_AMP:.2f}  (SM: m_s/m_d ~ 20)")
print(f"  Isospin1 (a5=1, SPEC5=2): {r4_iso1:.4f}  -> R4^x = {r4_iso1**X_AMP:.2f}")
print(f"  Isospin3 (a5=3, SPEC5=2): {r4_iso3:.4f}  -> R4^x = {r4_iso3**X_AMP:.2f}")

# Symmetry check: a5=1 and a5=3 should be related
print(f"\n  Isospin symmetry: ratio(a5=1)/ratio(a5=3) = {r4_iso1/r4_iso3:.4f}")
print(f"  (NB62 predicts Im2(a5=1)+Im2(a5=3)=0, so dynamical similarity expected)")

# Average isospin prediction
r4_iso_avg = (r4_iso1 + r4_iso3) / 2
pred_iso = r4_iso_avg ** X_AMP
print(f"\n  Isospin average R4 ratio: {r4_iso_avg:.4f}")
print(f"  Isospin R4^x prediction:  {pred_iso:.2f}")
print(f"  SM target m_c/m_u:        {M_C_MU_CURRENT}")

if pred_iso > 0 and r4_phys**X_AMP > 0:
    up_down_ratio = pred_iso / (r4_phys ** X_AMP)
    print(f"\n  Predicted (m_c/m_u)/(m_s/m_d) = {up_down_ratio:.2f}")
    print(f"  SM value  (m_c/m_u)/(m_s/m_d) = {M_C_MU_CURRENT/M_S_MD:.1f}")

# R3 ratio analysis (SPEC5 affects E2 which involves R3 via p=5)
print(f"\n\nR3 conjugate pair ratios (p=5 level — where SPEC5 acts):")
r3_phys = sector_ratios[0]['QUARK'][2]
r3_iso1 = sector_ratios[1]['QUARK'][2]
r3_iso3 = sector_ratios[3]['QUARK'][2]
print(f"  Physical (a5=0): {r3_phys:.4f}")
print(f"  Isospin1 (a5=1): {r3_iso1:.4f}")
print(f"  Isospin3 (a5=3): {r3_iso3:.4f}")

# Does R3^SPEC5_diff explain the up/down ratio?
spec5_diff = SPEC5[1] - SPEC5[0]  # = 2
r3_factor = r3_phys ** spec5_diff if r3_phys > 1 else r3_phys
print(f"\n  SPEC5 difference (iso vs phys): {spec5_diff}")
print(f"  R3_phys^{spec5_diff} = {r3_phys:.4f}^{spec5_diff} = {r3_factor:.4f}")
print(f"  If m_c/m_u ~ (m_s/m_d) * R3^SPEC5_diff:")
print(f"    = {r4_phys**X_AMP:.2f} * {r3_factor:.4f} = {r4_phys**X_AMP * r3_factor:.2f}")

TEST 3: ISOSPIN SECTOR MASS PREDICTION

Quark R4 conjugate pair ratios:
  Physical (a5=0, SPEC5=0): 1.4794  -> R4^x = 19.92  (SM: m_s/m_d ~ 20)
  Isospin1 (a5=1, SPEC5=2): 0.9933  -> R4^x = 0.95
  Isospin3 (a5=3, SPEC5=2): 0.5629  -> R4^x = 0.01

  Isospin symmetry: ratio(a5=1)/ratio(a5=3) = 1.7647
  (NB62 predicts Im2(a5=1)+Im2(a5=3)=0, so dynamical similarity expected)

  Isospin average R4 ratio: 0.7781
  Isospin R4^x prediction:  0.15
  SM target m_c/m_u:        588.0

  Predicted (m_c/m_u)/(m_s/m_d) = 0.01
  SM value  (m_c/m_u)/(m_s/m_d) = 29.4


R3 conjugate pair ratios (p=5 level — where SPEC5 acts):
  Physical (a5=0): 6.0881
  Isospin1 (a5=1): 1.0114
  Isospin3 (a5=3): 0.3841

  SPEC5 difference (iso vs phys): 2
  R3_phys^2 = 6.0881^2 = 37.0652
  If m_c/m_u ~ (m_s/m_d) * R3^SPEC5_diff:
    = 19.92 * 37.0652 = 738.37


## Test 4: Tower-Protected Sector (Neutrino Channel)

The $a_5 = 2$ sector has SPEC5 = 4 (highest) but is **tower-protected**: NB62 showed zero inter-generation splitting for this sector. If it maps to neutrinos, we expect near-degenerate conjugate pair ratios — the R₄ ratio should be close to 1.

This test checks whether the dynamics confirms the algebraic prediction of tower protection.

In [6]:
# -- Test 4: Tower-Protected Sector --
print("TEST 4: TOWER-PROTECTED SECTOR (a5=2, SPEC5=4)")
print("=" * 80)

print(f"\nR4 conjugate pair ratios across all sectors:")
print(f"{'Sector':<20} {'Lepton R4 ratio':>16} {'Quark R4 ratio':>16}")
print("-" * 55)
for a5i in range(4):
    tag = f"a5={a5i} (E5={SPEC5[a5i]})"
    lr = sector_ratios[a5i]['LEPTON'][3]
    qr = sector_ratios[a5i]['QUARK'][3]
    print(f"{tag:<20} {lr:16.4f} {qr:16.4f}")

r4_tp_lep = sector_ratios[2]['LEPTON'][3]
r4_tp_qrk = sector_ratios[2]['QUARK'][3]

print(f"\nTower-protected (a5=2) analysis:")
print(f"  Lepton R4 ratio: {r4_tp_lep:.6f}  (|1 - ratio| = {abs(1 - r4_tp_lep):.6f})")
print(f"  Quark  R4 ratio: {r4_tp_qrk:.6f}  (|1 - ratio| = {abs(1 - r4_tp_qrk):.6f})")

# Compare flatness across all 4 levels
print(f"\n  All-level profile for a5=2:")
print(f"  {'':>10} {'R1':>8} {'R2':>8} {'R3':>8} {'R4':>8}")
for pname in ['LEPTON', 'QUARK']:
    ratios = sector_ratios[2][pname]
    print(f"  {pname:>10}: " + "  ".join(f"{r:7.4f}" for r in ratios))
    # Flatness measure: std(log(ratio)) / mean(log(ratio))
    log_r = [np.log(r) for r in ratios if r > 0]
    if log_r:
        flatness = np.std(log_r) / (np.mean(np.abs(log_r)) + 1e-10)
        print(f"  {'':>10}  log-ratio flatness: {flatness:.4f} (0 = perfectly flat)")

# Compare to physical sector flatness
print(f"\n  Flatness comparison (std of log-ratios across levels):")
for a5i in range(4):
    for pname in ['LEPTON', 'QUARK']:
        ratios = sector_ratios[a5i][pname]
        log_r = [abs(np.log(r)) for r in ratios if r > 0]
        spread = np.std(log_r)
        mean_split = np.mean(log_r)
        tag = f"a5={a5i} {pname}"
        print(f"  {tag:<18}  mean|log(r)|={mean_split:.4f}  std={spread:.4f}")

TEST 4: TOWER-PROTECTED SECTOR (a5=2, SPEC5=4)

R4 conjugate pair ratios across all sectors:
Sector                Lepton R4 ratio   Quark R4 ratio
-------------------------------------------------------
a5=0 (E5=0)                    1.9795           1.4794
a5=1 (E5=2)                    1.0006           0.9933
a5=2 (E5=4)                    0.1532           1.0004
a5=3 (E5=2)                    1.0449           0.5629

Tower-protected (a5=2) analysis:
  Lepton R4 ratio: 0.153189  (|1 - ratio| = 0.846811)
  Quark  R4 ratio: 1.000372  (|1 - ratio| = 0.000372)

  All-level profile for a5=2:
                   R1       R2       R3       R4
      LEPTON:  0.0474   0.0363   0.1261   0.1532
              log-ratio flatness: 0.2390 (0 = perfectly flat)
       QUARK:  1.0003   1.0016   1.0013   1.0004
              log-ratio flatness: 0.6610 (0 = perfectly flat)

  Flatness comparison (std of log-ratios across levels):
  a5=0 LEPTON         mean|log(r)|=1.4459  std=0.4660
  a5=0 QUARK        

## Test 5: Multi-Level Bridge — SPEC5 Enhancement

NB70 showed that the single-level bridge $R_4^x$ works for the physical sector. The a₅≠0 sectors have extra SPEC5 eigenvalue contributions. Test whether combining R-levels weighted by tower eigenvalue differences captures the up/down mass ratio:

$$\frac{m_c/m_u}{m_s/m_d} \stackrel{?}{\sim} \left(\frac{R_{3,\text{iso}}}{R_{3,\text{phys}}}\right)^{\Delta E_5}$$

where $\Delta E_5 = \text{SPEC5}[1] - \text{SPEC5}[0] = 2$.

In [7]:
# -- Test 5: Multi-Level Bridge with SPEC5 Enhancement --
print("TEST 5: MULTI-LEVEL BRIDGE")
print("=" * 80)

# Compare R profiles between sectors
print(f"\nQuark conjugate pair ratios — all levels, all sectors:")
print(f"{'Sector':<18} {'R1':>8} {'R2':>8} {'R3':>8} {'R4':>8}")
print("-" * 50)
for a5i in range(4):
    tag = f"a5={a5i} (E5={SPEC5[a5i]})"
    vals = sector_ratios[a5i]['QUARK']
    print(f"{tag:<18} " + "  ".join(f"{v:7.4f}" for v in vals))

# Cross-sector R3 ratio (p=5 is where SPEC5 lives)
print(f"\n--- R3 cross-sector ratio (quark CP-active pair) ---")
r3_base = sector_ratios[0]['QUARK'][2]
for a5i in range(4):
    r3 = sector_ratios[a5i]['QUARK'][2]
    ratio = r3 / r3_base if r3_base > 0 else 0
    print(f"  R3(a5={a5i}) / R3(a5=0) = {r3:.4f} / {r3_base:.4f} = {ratio:.4f}")

# Cross-sector R4 ratio
print(f"\n--- R4 cross-sector ratio (quark CP-active pair) ---")
r4_base = sector_ratios[0]['QUARK'][3]
for a5i in range(4):
    r4 = sector_ratios[a5i]['QUARK'][3]
    ratio = r4 / r4_base if r4_base > 0 else 0
    print(f"  R4(a5={a5i}) / R4(a5=0) = {r4:.4f} / {r4_base:.4f} = {ratio:.4f}")

# Multi-level bridge: R3^SPEC5 * R4^x
print(f"\n--- Composite prediction: R4^x * (cross-sector R3 enhancement) ---")
for a5i in range(4):
    r4_pred = sector_ratios[a5i]['QUARK'][3] ** X_AMP
    # Also try: physical sector R4 prediction, then enhance with cross-sector R3
    r3_ratio = sector_ratios[a5i]['QUARK'][2] / r3_base if r3_base > 0 else 1
    enhanced = (r4_base ** X_AMP) * (r3_ratio ** SPEC5[a5i])
    tag = f"a5={a5i} (E5={SPEC5[a5i]})"
    print(f"  {tag:<18} R4^x = {r4_pred:>10.2f}   enhanced = {enhanced:>10.2f}")

# Direct approach: does R4_iso^x give m_c/m_u directly?
print(f"\n--- Direct amplification per sector ---")
for a5i in range(4):
    for pname in ['LEPTON', 'QUARK']:
        pred = sector_pred[a5i][pname]
        print(f"  a5={a5i} {pname:<8}: R4_ratio^x = {pred:>10.2f}")

TEST 5: MULTI-LEVEL BRIDGE

Quark conjugate pair ratios — all levels, all sectors:
Sector                   R1       R2       R3       R4
--------------------------------------------------
a5=0 (E5=0)        36.7511  20.1672   6.0881   1.4794
a5=1 (E5=2)         1.0051   1.0112   1.0114   0.9933
a5=2 (E5=4)         1.0003   1.0016   1.0013   1.0004
a5=3 (E5=2)         0.1331   0.1785   0.3841   0.5629

--- R3 cross-sector ratio (quark CP-active pair) ---
  R3(a5=0) / R3(a5=0) = 6.0881 / 6.0881 = 1.0000
  R3(a5=1) / R3(a5=0) = 1.0114 / 6.0881 = 0.1661
  R3(a5=2) / R3(a5=0) = 1.0013 / 6.0881 = 0.1645
  R3(a5=3) / R3(a5=0) = 0.3841 / 6.0881 = 0.0631

--- R4 cross-sector ratio (quark CP-active pair) ---
  R4(a5=0) / R4(a5=0) = 1.4794 / 1.4794 = 1.0000
  R4(a5=1) / R4(a5=0) = 0.9933 / 1.4794 = 0.6714
  R4(a5=2) / R4(a5=0) = 1.0004 / 1.4794 = 0.6762
  R4(a5=3) / R4(a5=0) = 0.5629 / 1.4794 = 0.3805

--- Composite prediction: R4^x * (cross-sector R3 enhancement) ---
  a5=0 (E5=0)        R4^x =

In [8]:
# -- NB71 Scorecard --
print("NB71 SCORECARD")
print("=" * 65)

# Assess results
# Identity #129: Sector differentiation — do a5 sectors differ dynamically?
r4_phys_q = sector_ratios[0]['QUARK'][3]
r4_iso1_q = sector_ratios[1]['QUARK'][3]
r4_iso3_q = sector_ratios[3]['QUARK'][3]
r4_tp_q   = sector_ratios[2]['QUARK'][3]

print(f"\n#129 CHARGE SECTOR DIFFERENTIATION")
print(f"  Physical (a5=0) quark R4 ratio:      {r4_phys_q:.4f}")
print(f"  Isospin1 (a5=1) quark R4 ratio:      {r4_iso1_q:.4f}")
print(f"  Isospin3 (a5=3) quark R4 ratio:      {r4_iso3_q:.4f}")
print(f"  Tower-P  (a5=2) quark R4 ratio:      {r4_tp_q:.4f}")
all_r4 = [r4_phys_q, r4_iso1_q, r4_iso3_q, r4_tp_q]
spread = (max(all_r4) - min(all_r4)) / np.mean(all_r4)
print(f"  Spread: {spread*100:.1f}%")
if spread > 0.05:
    print(f"  Verdict: PASS — sectors are dynamically differentiated ({spread*100:.1f}% spread)")
else:
    print(f"  Verdict: NULL — sectors too similar ({spread*100:.1f}% spread)")

# Identity #130: Isospin degeneracy — are a5=1 and a5=3 symmetric?
print(f"\n#130 ISOSPIN DOUBLET SYMMETRY")
iso_sym = abs(r4_iso1_q - r4_iso3_q) / ((r4_iso1_q + r4_iso3_q)/2) * 100
print(f"  |R4(a5=1) - R4(a5=3)| / mean = {iso_sym:.2f}%")
if iso_sym < 5:
    print(f"  Verdict: PASS — isospin doublet near-degenerate ({iso_sym:.2f}%)")
else:
    print(f"  Verdict: NULL — significant isospin splitting ({iso_sym:.2f}%)")

# Identity #131: Tower protection — is a5=2 flat?
print(f"\n#131 TOWER PROTECTION (a5=2)")
r4_tp_lep = sector_ratios[2]['LEPTON'][3]
r4_tp_qrk = sector_ratios[2]['QUARK'][3]
tp_lep_split = abs(np.log(r4_tp_lep))
tp_qrk_split = abs(np.log(r4_tp_qrk))
phys_lep_split = abs(np.log(sector_ratios[0]['LEPTON'][3]))
phys_qrk_split = abs(np.log(sector_ratios[0]['QUARK'][3]))
lep_reduction = tp_lep_split / phys_lep_split if phys_lep_split > 0 else 0
qrk_reduction = tp_qrk_split / phys_qrk_split if phys_qrk_split > 0 else 0
print(f"  Tower-protected |log(R4)| vs physical sector:")
print(f"    Lepton: {tp_lep_split:.4f} vs {phys_lep_split:.4f} (ratio: {lep_reduction:.3f})")
print(f"    Quark:  {tp_qrk_split:.4f} vs {phys_qrk_split:.4f} (ratio: {qrk_reduction:.3f})")
if lep_reduction < 0.5 and qrk_reduction < 0.5:
    print(f"  Verdict: PASS — a5=2 shows >50% reduction in generation splitting")
elif lep_reduction < 0.8 or qrk_reduction < 0.8:
    print(f"  Verdict: PARTIAL — some suppression but not dramatic")
else:
    print(f"  Verdict: NULL — no clear tower protection in dynamics")

# Identity #132: Up/down mass ratio from sector R4
print(f"\n#132 UP-TYPE vs DOWN-TYPE MASS RATIO")
iso_avg_r4 = (r4_iso1_q + r4_iso3_q) / 2
pred_up = iso_avg_r4 ** X_AMP
pred_down = r4_phys_q ** X_AMP
if pred_down > 0:
    ratio_ud = pred_up / pred_down
    print(f"  Isospin R4^x / Physical R4^x = {pred_up:.2f} / {pred_down:.2f} = {ratio_ud:.2f}")
    print(f"  SM (m_c/m_u)/(m_s/m_d) = {M_C_MU_CURRENT/M_S_MD:.1f}")
    dev = abs(ratio_ud - M_C_MU_CURRENT/M_S_MD) / (M_C_MU_CURRENT/M_S_MD) * 100
    if dev < 50:
        print(f"  Verdict: PASS — correct order of magnitude ({dev:.0f}% off)")
    elif dev < 200:
        print(f"  Verdict: NULL — right direction, wrong scale ({dev:.0f}% off)")
    else:
        print(f"  Verdict: NULL — does not match ({dev:.0f}% off)")
else:
    print(f"  Cannot compute (pred_down = 0)")

print()
print("-" * 65)
n_pass = sum(1 for x in [spread > 0.05, iso_sym < 5,
             (lep_reduction < 0.5 and qrk_reduction < 0.5)]
             if x)
print(f"NB71: {n_pass + (1 if pred_down > 0 and dev < 50 else 0)} identities assessed")
print(f"Running total: 128 + NB71 new = predictions/identities, 0 free parameters")
print(f"\nScope: This notebook maps the LANDSCAPE of charge sector dynamics.")
print(f"Whether the isospin sector directly predicts m_c/m_u depends on")
print(f"whether R4^x applies uniformly across sectors or needs sector-specific")
print(f"modification (tower eigenvalue weighting, R3 enhancement, etc.).")

NB71 SCORECARD

#129 CHARGE SECTOR DIFFERENTIATION
  Physical (a5=0) quark R4 ratio:      1.4794
  Isospin1 (a5=1) quark R4 ratio:      0.9933
  Isospin3 (a5=3) quark R4 ratio:      0.5629
  Tower-P  (a5=2) quark R4 ratio:      1.0004
  Spread: 90.8%
  Verdict: PASS — sectors are dynamically differentiated (90.8% spread)

#130 ISOSPIN DOUBLET SYMMETRY
  |R4(a5=1) - R4(a5=3)| / mean = 55.32%
  Verdict: NULL — significant isospin splitting (55.32%)

#131 TOWER PROTECTION (a5=2)
  Tower-protected |log(R4)| vs physical sector:
    Lepton: 1.8761 vs 0.6828 (ratio: 2.748)
    Quark:  0.0004 vs 0.3916 (ratio: 0.001)
  Verdict: PARTIAL — some suppression but not dramatic

#132 UP-TYPE vs DOWN-TYPE MASS RATIO
  Isospin R4^x / Physical R4^x = 0.15 / 19.92 = 0.01
  SM (m_c/m_u)/(m_s/m_d) = 29.4
  Verdict: NULL — right direction, wrong scale (100% off)

-----------------------------------------------------------------
NB71: 1 identities assessed
Running total: 128 + NB71 new = predictions/identiti